# Fusion des deux sources de données

Ce notebook fusionne les données des toilettes publiques et des urinoirs publics.

In [1]:
# Importation de pandas
import pandas as pd

# Chargement des deux fichiers CSV
df_toilettes = pd.read_csv('toilettes_publiques_vbx.csv', sep=';')
df_urinoirs = pd.read_csv('urinoirs-publics-vbx.csv', sep=';')

print(f"Toilettes: {df_toilettes.shape[0]} lignes")
print(f"Urinoirs: {df_urinoirs.shape[0]} lignes")

Toilettes: 116 lignes
Urinoirs: 22 lignes


## 7. Stratégie de fusion

**Nouvelle colonne** : "Type_equipement" pour distinguer urinoirs et toilettes

In [ ]:
# Ajout d'une colonne pour identifier le type d'équipement
df_toilettes['Type_equipement'] = 'Toilette'
df_urinoirs['Type_equipement'] = 'Urinoir'

# Colonnes communes aux deux datasets
colonnes_communes = ['objectid', 'Statut', 'Localisation', 'Adresse', 'Code postal', 
                     'Commune', 'Quartier', 'Google Maps', 'Geo Point', 'Geo Shape', 
                     'Type_equipement']

# Sélection des colonnes communes (presentes dans les deux)
cols_toilettes = [c for c in df_toilettes.columns if c in df_urinoirs.columns]
cols_urinoirs = [c for c in df_urinoirs.columns if c in df_toilettes.columns]

print("Colonnes communes aux deux datasets:")
print(set(cols_toilettes)  & set(cols_urinoirs))

Colonnes communes aux deux datasets:
{'Geo Shape', 'Type.1', 'Adresse', 'Statut', 'Google Maps', 'Gemeente', 'Tarification', 'Category', 'Categorie', 'Catégorie', 'Localisation', 'Adres', 'Tarieven', 'Gestionnaire', 'Beheerder', 'Code postal', "Heures d'ouverture", 'Quartier', 'Type_equipement', 'Commune', 'Type.2', 'Type', 'Wijk', 'Google Street View', 'objectid', 'Pricing', 'Geo Point'}


## 8. Type de jointure et gestion des doublons

**Type de jointure** : concaténation (union) car on veut tous les équipements

Les deux datasets ont des structures différentes, on utilise `pd.concat()` avec outer join.

In [3]:
# Fusion par concaténation (tous les enregistrements des deux sources)
df_fusionne = pd.concat([df_toilettes, df_urinoirs], ignore_index=True)

print(f"Total après fusion: {df_fusionne.shape[0]} lignes")

# Vérification des doublons sur objectid
doublons = df_fusionne['objectid'].duplicated().sum()
print(f"Doublons sur objectid: {doublons}")

# Suppression des doublons si existants
if doublons > 0:
    df_fusionne = df_fusionne.drop_duplicates(subset=['objectid'])
    print(f"Après suppression: {df_fusionne.shape[0]} lignes")

Total après fusion: 138 lignes
Doublons sur objectid: 0


## 9. Statistiques finales

In [4]:
# Nombre total d'équipements
total = df_fusionne.shape[0]
print(f"Total équipements sanitaires: {total}")

# Répartition par type
repartition = df_fusionne['Type_equipement'].value_counts()
print("\nRépartition:")
print(repartition)

# Proportion urinoirs
nb_urinoirs = (df_fusionne['Type_equipement'] == 'Urinoir').sum()
proportion = (nb_urinoirs / total) * 100
print(f"\nProportion urinoirs: {proportion:.1f}%")

Total équipements sanitaires: 138

Répartition:
Type_equipement
Toilette    116
Urinoir      22
Name: count, dtype: int64

Proportion urinoirs: 15.9%
